<a href="https://colab.research.google.com/github/suriarasai/BEAD2026/blob/main/colab/rebu/00LoadData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚕 REBU Taxi ETL/ELT with PySpark UDFs
Read the case study shared in classroom exercises carefully. This notebook details the sample data. It is followed by set of 6 other colab notebooks. Each notebook will walk you through systematic ways of generating insights consumed by various stakeholders using ETL and ELT pipelines.

### 00 · Data and Notebooks

Six Colab notebooks that turn the taxi sample data (bookings, payments, rewards, passengers, drivers, fleet, places) into **business-intelligence outcomes**, each built around PySpark user-defined functions (UDFs).

**What you get across the series**
* ~25 UDFs covering every UDF style: Python row UDFs, struct/array returns, UDF factories, pandas Series UDFs, pandas grouped-aggregate UDFs, `applyInPandas` grouped-map models, and UDFs **registered for Spark SQL** (ELT).
* A **bronze → silver → gold lakehouse** written to your Drive, ending in a **BI-ready star schema** (Parquet + CSV).
* Real findings in *this* data: a truncated booking extract, a vehicle-class-blind fare engine, unaligned loyalty tiers and impossible overlapping trips, each sized in dollars or percent.

## Load the Data and Run the Notebooks
1. **Put the 9 CSVs in one Google Drive folder**, e.g. `My Drive/sg_taxi_data/`
   (`Booking.csv`, `Payment.csv`, `Rewards.csv`, `Passengers.csv`, `Drivers.csv`, `Taxi.csv`, `TaxiType.csv`, `Place.csv`, `PaymentType.csv`). Feel free to rename the data files if necessary.
2. **Open any notebook in Colab:** *Navigate to File in Git → Open in colab notebook*, or save the `.ipynb` files to Drive and choose *Open with → Google Colaboratory*.
3. If your folder isn't `rebu`, edit `DATA_DIR` in setup cell **Point the Data** step. Then **Runtime → Run all** and approve the Drive prompt.

Each notebook is **self-contained** (its own setup and loading), so you can demo any one in isolation. On a free Colab runtime each takes about 2–4 minutes. If Drive isn't available, setup cell 2 falls back to a file-upload widget.

## Summary of Notebooks
| Notebook | Business outcome | UDF techniques shown |
|---|---|---|
| **01 · Ingestion, Cleansing & Data Quality** | Trusted, PDPA-safe silver layer + DQ scorecard | flexible timestamp parser (with built-in benchmark), phone normaliser → `struct`, address parser + geo consistency, PII masking (pandas UDF), **configurable DQ rule engine** (UDF factory → `array`) |
| **02 · Trip Enrichment & Pricing** | Geo/time-tagged trips, **pricing-compliance & revenue-leakage** view, OD analytics | place classifier, postcode → district/region, haversine row UDF **vs pandas UDF** benchmark, config-driven tariff/holiday UDF, **expected-fare UDF factory** |
| **03 · Customer Intelligence** | Customer-360: RFM segments, personas, churn trend, loyalty audit, re-tiering | **pandas grouped-aggregate** UDFs, RFM segment mapper, **`applyInPandas`** behaviour profiles, z-score persona factory, loyalty-rule factory, value-tier factory |
| **04 · Driver & Fleet Performance** | Driver scorecard, overlapping-trip integrity check, fleet lifecycle & **CO₂/ESG** | `applyInPandas` **sessionisation** (shifts, fatigue, overlaps), composite score factory with global bounds, vehicle-profile `struct`, CO₂ pandas UDF |
| **05 · Payments, Revenue & Tax (ELT in SQL)** | Revenue waterfall, **date-effective GST** split & quarterly payable, channel economics, no-show fee sizing | Python & pandas UDFs **registered as SQL functions**; layered SQL models (staging → intermediate → mart) |
| **06 · Forecasting, Anomalies & Star Schema** | Per-region demand forecast & back-test, anomaly feed, **star schema export** for Power BI / Looker / Tableau | one model per group via `applyInPandas`, robust-z anomaly detector, calendar-dimension UDF, FK integrity tests |

## Choosing the right kind of UDF
| Need | Use | Example in this series |
|---|---|---|
| Logic Spark already has (casting, date parsing, regex, hashing) | **Built-in functions**, not a UDF | `try_to_timestamp`, `regexp_extract`, `sha2` (NB01 benchmark: built-in ~5× faster) |
| Row-level business rules, branching, lookups | **Python UDF** (`@F.udf`) | `rfm_segment`, `payment_outcome`, `place_category` |
| Several outputs from one call | Python UDF returning **`StructType`** | `normalize_sg_phone`, `parse_sg_address`, `vehicle_profile` |
| Variable-length results | UDF returning **`ArrayType`** | `dq_issues` |
| Rules/thresholds business users should tune | **UDF factory** (closure over a config dict) | `make_dq_udf`, `make_expected_fare`, `make_points_udf` |
| Vectorisable numeric/string maths | **pandas UDF** (Series → Series) | `haversine_vec`, `mask_email`, `co2_kg` |
| Custom aggregate per group | **pandas grouped-agg UDF** (Series → scalar) | `mode_str`, `gap_regularity` |
| Whole-group logic: sorting, sessionising, fitting a model | **`groupBy().applyInPandas`** | `driver_timeline`, `rider_features`, `forecast_region` |
| Analysts working in SQL (ELT) | **`spark.udf.register`** | `gst_split`, `fiscal_period`, `net_of_gst_vec` |

**Pandas-UDF Practices:** a pandas UDF sees one Arrow *batch*, not the whole column. Compute global statistics (min/max, mean/sd) first and close over them (NB03 personas, NB04 driver score).

## Synthetic Dataset and Exploratory Analysis
| # | Finding | Evidence | Notebook |
|---|---|---|---|
| 1 | **Booking extract is truncated** at BookingID 505,630; Payment & Rewards cover ~1M bookings | 475,664 payments (49%) and 460,175 rewards have no booking; summing Payment alone doubles revenue (S$15.5M vs S$7.85M matched) | 01 |
| 2 | **Relational data-quality defects** | 20.5% of trips predate the passenger's registration, 9.7% predate the driver's join date, 3.3% overlap the same driver's previous trip; 69.5% of bookings are fully clean | 01, 04 |
| 3 | **Fares ignore vehicle class** | Same fitted S$/km for every class; vs the TaxiType tariff, Limo under-recovers **S$577k (−26%)** and Standard is over-charged **S$358k (+26%)** | 02 |
| 4 | `Distance` is **straight-line**, not driven km | Matches haversine between places (r = 1.00) | 02 |
| 5 | Tariff bands recovered from data | ×1.5 for 00:00–06:59; ×1.25 for 07:00–09:59 & 17:00–20:59 (by booking time) | 02 |
| 6 | **Loyalty ledger reconciles 100%** | 470,989 eligible bookings → 7.44M points, exact; 9,540 completed trips with failed payment correctly earn none | 03 |
| 7 | **MemberCategory isn't value-aligned** | Avg 12-month spend ≈ S$3.87k in A, B and C alike; 223 customers' category contradicts their value | 03 |
| 8 | Revenue waterfall | S$8.24M booked → S$7.69M collected → **S$7.09M net of GST**; S$156k lost to failed payments; 71% of cancellations unbilled; no-shows never charged | 05 |
| 9 | Fleet renewal & ESG | 75 taxis past an assumed 8-year life and 85 more due within a year; hybrids do 14% of km; swapping the top-50 petrol taxis saves ≈21 t CO₂/yr (illustrative factors) | 04 |
| 10 | Passenger postcodes unreliable | 19.9% use non-existent postal sectors; most valid ones disagree with the street's region | 01 |

The data is synthetic and very uniform (flat demand by hour, weekday and payment method), so the notebooks lean on **reconciliation, pricing and integrity** stories, where the data does carry signal. The segmentation and forecasting code is production-shaped and will show far more contrast on real data.

## Source data model
```
Passengers(PassengerID) ─┐                       ┌─ Taxi(TaxiID) ── TaxiType(TaxiTypeID)
Drivers(DriverID) ───────┼── Booking(BookingID) ─┤
Place(PlaceID) ──pickup/dropoff──┘     │         └─ (DriverID = TaxiID: 1-to-1 pairing)
                                       ├── Payment(PaymentID) ── PaymentType(PaymentTypeID)
                                       └── Rewards(RewardID)  — BookingID parsed from "Earned from booking #N"
```

## Environment & data check
Run these cells to confirm Spark starts and all nine files are found before demoing the other notebooks.

In [3]:
#@title  1 · Environment (PySpark + Java) — run once { display-mode: "form" }
import os, sys, re, glob, time, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

def _java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
        major = int(m.group(1))
        return int(m.group(2)) if major == 1 else major
    except Exception:
        return None

try:
    import pyspark
except ImportError:                      # recent Colab images ship PySpark; install only if missing
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.5"], check=True)
    import pyspark

_spark_major, _jv = int(pyspark.__version__.split(".")[0]), _java_major()
# Spark 3.5 runs on Java 8/11/17, Spark 4.x on Java 17/21 → fix the JDK on Colab if they don't match
if IN_COLAB and (_jv is None or (_spark_major < 4 and _jv > 17) or (_spark_major >= 4 and _jv < 17)):
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(f"PySpark {pyspark.__version__} · Java {_java_major()} · Colab={IN_COLAB}")

PySpark 4.0.4 · Java 21 · Colab=True


In [4]:
#@title  2 · Point to the data { display-mode: "form" }
#@markdown Put the 9 CSV files (Booking, Payment, Rewards, Passengers, Drivers, Taxi, TaxiType, Place, PaymentType)
#@markdown in one Google Drive folder and set its path here. File names may carry a prefix (e.g. `2eaaf3ac-Passengers.csv`).
DATA_DIR = "/content/drive/MyDrive/rebu"  #@param {type:"string"}
TABLES = ["Booking", "Payment", "Rewards", "Passengers", "Drivers", "Taxi", "TaxiType", "Place", "PaymentType"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not glob.glob(os.path.join(DATA_DIR, "*.csv")):
        print(f"No CSVs in {DATA_DIR} – falling back to direct upload (Drive is faster for the ~50 MB files).")
        from google.colab import files
        DATA_DIR = "/content/data"; os.makedirs(DATA_DIR, exist_ok=True)
        for fname, blob in files.upload().items():
            with open(os.path.join(DATA_DIR, fname), "wb") as fh:
                fh.write(blob)
else:
    DATA_DIR = os.environ.get("SG_TAXI_DATA", "./data")

LAKE_DIR = os.path.join(DATA_DIR, "lakehouse")      # bronze / silver / gold outputs land here

def find_csv(name):
    """Match 'Taxi.csv' or '<prefix>-Taxi.csv' but never 'TaxiType.csv'."""
    hits = [p for p in glob.glob(os.path.join(DATA_DIR, "*.csv"))
            if re.fullmatch(rf"(?:.*[-_ ])?{name}\.csv", os.path.basename(p), flags=re.I)]
    if not hits:
        raise FileNotFoundError(f"{name}.csv not found in {DATA_DIR}")
    return hits[0]

for t in TABLES:
    print(f"{t:<12} → {os.path.basename(find_csv(t))}")

Mounted at /content/drive
Booking      → Booking.csv
Payment      → Payment.csv
Rewards      → Rewards.csv
Passengers   → Passengers.csv
Drivers      → Drivers.csv
Taxi         → Taxi.csv
TaxiType     → TaxiType.csv
Place        → Place.csv
PaymentType  → PaymentType.csv


In [5]:
#@title  3 · Spark session, readers & chart style { display-mode: "form" }
from pyspark.sql import SparkSession, Window, functions as F, types as T
import pandas as pd, numpy as np
import matplotlib as mpl, matplotlib.pyplot as plt
from IPython.display import display, Markdown

# --add-opens lets Arrow (used by pandas UDFs) work on newer JDKs; harmless on Java 11
_JOPTS = " ".join(f"--add-opens=java.base/{p}=ALL-UNNAMED"
                  for p in ["java.nio", "sun.nio.ch", "java.lang", "java.util"]) + " -Dio.netty.tryReflectionSetAccessible=true"
spark = (SparkSession.builder.appName("SG-Taxi-BI-UDFs").master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.session.timeZone", "Asia/Singapore")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.driver.extraJavaOptions", _JOPTS)
         .config("spark.executor.extraJavaOptions", _JOPTS)
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def read_csv(name):
    """Bronze read: every column as a raw string, delimiter sniffed (Booking.csv is tab-separated)."""
    path = find_csv(name)
    with open(path, encoding="utf-8-sig") as fh:
        header = fh.readline()
    sep = "\t" if header.count("\t") > header.count(",") else ","
    return spark.read.csv(path, header=True, sep=sep, inferSchema=False, quote='"', escape='"')

def show(df, n=10):
    """Pretty table in Colab (Spark → pandas for display only)."""
    display(df.limit(n).toPandas())

def timed(label, df):
    """Force full evaluation without collecting (noop sink) and time it."""
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    secs = time.perf_counter() - t0
    print(f"{label:<48}{secs:7.2f} s")
    return secs

# Chart style: thin marks, recessive grid, fixed categorical order, single-hue sequential ramp
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQ = mpl.colors.LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
mpl.rcParams.update({
    "figure.figsize": (10, 4.2), "figure.dpi": 110, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.prop_cycle": mpl.cycler(color=PALETTE), "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left", "lines.linewidth": 2,
    "legend.frameon": False, "font.size": 10, "axes.axisbelow": True,
})
print("Spark", spark.version, "ready")

Spark 4.0.4 ready


In [6]:
for t in TABLES:
    df = read_csv(t)
    print(f"{t:<12}{df.count():>10,} rows · {len(df.columns)} cols")
print("\nAll set: open 01IngestionCleansingDataQuality.ipynb next.")

Booking        505,630 rows · 12 cols
Payment        962,068 rows · 6 cols
Rewards        931,164 rows · 5 cols
Passengers       1,000 rows · 10 cols
Drivers            400 rows · 11 cols
Taxi               400 rows · 6 cols
TaxiType             5 rows · 5 cols
Place               87 rows · 5 cols
PaymentType          6 rows · 2 cols

All set: open 01_Ingestion_Cleansing_DataQuality.ipynb next.
